In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

In [20]:
# Load the textfile
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

# Split document into chunks
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n",". "," ",""],
    chunk_size = 500,chunk_overlap = 50
)
docs = splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is an open-source framework designed to simplify the construction of applications powered by\nlarge language models (LLMs), serving as the easiest starting point for developers while remaining \nflexible and production-ready. Its core philosophy rests on the belief that LLMs become far more powerful\nwhen combined with external data sources, and that future applications will become increasingly agentic—capable'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='of reasoning and acting autonomously. LangChain addresses two central challenges: standardizing model \ninputs and outputs across different providers so developers can easily switch to the latest state-of-the-art\nmodels without vendor lock-in, and making it simple to use models for orchestrating complex flows that interact\nwith other data and computation through dynamically defined tools.The framework provides four main building'

In [21]:
# User query
query = "How can i use langchain to build application with memory and tools?"

In [22]:
# FAISS and HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstores = FAISS.from_documents(docs,embedding_model)
retriever = vectorstores.as_retriever(
    search_kwargs={"k":8}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f41fd31d6a0>, search_kwargs={'k': 8})

In [24]:
from langchain_groq import ChatGroq

llm = ChatGroq(
     model="openai/gpt-oss-20b",
     temperature=0.7,
     api_key=os.getenv("GROQ_API_KEY")
 )

In [25]:
# Prompt template
prompt = PromptTemplate.from_template("""
you are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

user question: "{question}"
Documents: {documents}

Instructions:
 - Think about the relevance of each document to the users question.
 - Return a list of document indices in ranked order, starting from the most relevant.

output format: Comma-separated document indices(e.g., 2,1,3,0...)"""
)

In [26]:
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='870ea92c-ba26-48f3-9291-2fbafea33f6f', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is an open-source framework designed to simplify the construction of applications powered by\nlarge language models (LLMs), serving as the easiest starting point for developers while remaining \nflexible and production-ready. Its core philosophy rests on the belief that LLMs become far more powerful\nwhen combined with external data sources, and that future applications will become increasingly agentic—capable'),
 Document(id='5289861b-35c3-49f3-bcfe-45a6ca7927f4', metadata={'source': 'langchain_sample.txt'}, page_content='of reasoning and acting autonomously. LangChain addresses two central challenges: standardizing model \ninputs and outputs across different providers so developers can easily switch to the latest state-of-the-art\nmodels without vendor lock-in, and making it simple to use models for orchestrating complex flows that interact\nwith other data and co

In [27]:
chain = prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nyou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nuser question: "{question}"\nDocuments: {documents}\n\nInstructions:\n - Think about the relevance of each document to the users question.\n - Return a list of document indices in ranked order, starting from the most relevant.\n\noutput format: Comma-separated document indices(e.g., 2,1,3,0...)')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': T

In [28]:
doc_lines = [f"{i+1}.{doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [29]:
response = chain.invoke({"question":query,"documents":formatted_docs})
response

'7,2,1,3,6,8,4,5'

In [30]:
indices = [int(x.strip())-1 for x in response.split(",") if x.strip().isdigit()]
indices

[6, 1, 0, 2, 5, 7, 3, 4]

In [31]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[Document(id='40fd8b49-b1d5-4089-a337-3efe394d45ee', metadata={'source': 'langchain_sample.txt'}, page_content='like documents or databases, typically working with vector indices and embeddings to locate semantically similar \ncontent—this is what makes retrieval-augmented generation (RAG) possible. Tools represent external functionality \nthe LLM can invoke, such as querying databases, calling web APIs, or performing calculations, extending the system \nbeyond text generation.To understand how these pieces fit together in practice, consider the RAG pipeline, which'),
 Document(id='5289861b-35c3-49f3-bcfe-45a6ca7927f4', metadata={'source': 'langchain_sample.txt'}, page_content='of reasoning and acting autonomously. LangChain addresses two central challenges: standardizing model \ninputs and outputs across different providers so developers can easily switch to the latest state-of-the-art\nmodels without vendor lock-in, and making it simple to use models for orchestrating complex flows t

In [32]:
# Show results
print("\n Final reranked results: ")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}:\n{doc.page_content}")


 Final reranked results: 

Rank 1:
like documents or databases, typically working with vector indices and embeddings to locate semantically similar 
content—this is what makes retrieval-augmented generation (RAG) possible. Tools represent external functionality 
the LLM can invoke, such as querying databases, calling web APIs, or performing calculations, extending the system 
beyond text generation.To understand how these pieces fit together in practice, consider the RAG pipeline, which

Rank 2:
of reasoning and acting autonomously. LangChain addresses two central challenges: standardizing model 
inputs and outputs across different providers so developers can easily switch to the latest state-of-the-art
models without vendor lock-in, and making it simple to use models for orchestrating complex flows that interact
with other data and computation through dynamically defined tools.The framework provides four main building

Rank 3:
LangChain is an open-source framework designed to simplif